In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalogo", "catalog_dev")
catalogo = dbutils.widgets.get("catalogo").strip()
if catalogo not in ("catalog_dev", "catalog_prod"):
    raise ValueError("Solo se permiten catalog_dev o catalog_prod")

def silver(nombre):
    return spark.table(f"{catalogo}.silver.{nombre}")

def guardar_hecho(df, nombre, esperado):
    destino = f"{catalogo}.golden.{nombre}"
    columnas = spark.table(destino).columns
    faltantes = set(columnas) - set(df.columns)
    adicionales = set(df.columns) - set(columnas)
    if faltantes or adicionales:
        raise ValueError(f"{nombre}: columnas faltantes={faltantes}; adicionales={adicionales}")

    df.select(*columnas).write.mode("overwrite").insertInto(destino)
    total = spark.table(destino).count()
    if total != esperado:
        raise ValueError(f"{nombre}: esperado={esperado:,}, cargado={total:,}")
    print(f"OK | {destino}: {total:,} registros")

def exigir_clave_unica(df, columnas, origen):
    nulos = None
    for col in columnas:
        cond = F.col(col).isNull()
        nulos = cond if nulos is None else (nulos | cond)
    if df.filter(nulos).limit(1).count():
        raise ValueError(f"{origen}: clave nula en {columnas}")
    if df.groupBy(*columnas).count().filter(F.col("count") > 1).limit(1).count():
        raise ValueError(f"{origen}: clave duplicada en {columnas}")

pedidos = silver("pedidos")
clientes = silver("clientes")
detalle = silver("detalle_ventas")
productos = silver("productos")
pagos = silver("pagos")
resenas = silver("resenas")

for nombre, df, claves in [
    ("silver.pedidos", pedidos, ["id_pedido"]),
    ("silver.clientes", clientes, ["id_cliente"]),
    ("silver.productos", productos, ["id_producto"]),
    ("silver.detalle_ventas", detalle, ["id_pedido", "id_item"]),
    ("silver.pagos", pagos, ["id_pedido", "secuencia_pago"]),
    ("silver.resenas", resenas, ["id_resena", "id_pedido"]),
]:
    exigir_clave_unica(df, claves, nombre)

cabeceras = (
    pedidos.alias("p")
    .join(clientes.select("id_cliente", "id_cliente_unico", "codigo_postal").alias("c"),
          F.col("p.id_cliente") == F.col("c.id_cliente"), "left")
    .select(
        F.col("p.id_pedido"),
        F.col("p.estado_pedido"),
        F.col("p.fecha_compra"),
        F.col("p.fecha_entrega_cliente"),
        F.col("p.fecha_entrega_estimada"),
        F.col("c.id_cliente_unico"),
        F.col("c.codigo_postal").alias("codigo_postal_cliente")
    )
)


if cabeceras.filter(F.col("id_cliente_unico").isNull()).limit(1).count():
    raise ValueError("Hay pedidos sin cliente en Silver: revisar antes de cargar Gold")

def fecha_key(columna):
    return F.date_format(F.to_date(F.col(columna)), "yyyyMMdd").cast("int")

print("Prevalidaciones correctas: claves únicas y pedidos asociados a clientes")

In [0]:
productos_categoria = productos.select("id_producto", "categoria")

fact_ventas = (
    detalle.alias("d")
    .join(cabeceras.alias("c"), F.col("d.id_pedido") == F.col("c.id_pedido"), "left")
    .join(productos_categoria.alias("pr"), F.col("d.id_producto") == F.col("pr.id_producto"), "left")
    .select(
        F.col("d.id_pedido").alias("id_pedido"),
        F.col("d.id_item").alias("id_item"),
        fecha_key("c.fecha_compra").alias("id_fecha"),
        F.col("c.id_cliente_unico"),
        F.col("d.id_producto"),
        F.col("pr.categoria").alias("id_categoria"),
        F.col("d.id_vendedor"),
        F.col("c.codigo_postal_cliente"),
        F.col("c.estado_pedido"),
        F.col("d.precio"),
        F.col("d.valor_flete"),
        F.lit(1).cast("int").alias("cantidad_articulos")
    )
)

guardar_hecho(fact_ventas, "fact_ventas", 112650)

In [0]:
importes_pedido = (
    detalle.groupBy("id_pedido")
    .agg(
        F.count("*").cast("bigint").alias("cantidad_articulos"),
        F.sum("precio").cast("decimal(18,2)").alias("importe_articulos"),
        F.sum("valor_flete").cast("decimal(18,2)").alias("importe_flete")
    )
)

fact_pedidos = (
    cabeceras.alias("p")
    .join(importes_pedido.alias("a"), "id_pedido", "left")
    .select(
        F.col("id_pedido"),
        fecha_key("p.fecha_compra").alias("id_fecha"),
        F.col("p.id_cliente_unico"),
        F.col("p.codigo_postal_cliente"),
        F.col("p.estado_pedido"),
        F.col("p.fecha_compra"),
        F.col("p.fecha_entrega_cliente"),
        F.col("p.fecha_entrega_estimada"),
        ((F.col("p.fecha_entrega_cliente").cast("long") - F.col("p.fecha_compra").cast("long"))
          / F.lit(86400.0)).cast("double").alias("dias_entrega"),
        F.datediff(F.to_date("p.fecha_entrega_cliente"),
                   F.to_date("p.fecha_entrega_estimada")).cast("double").alias("dias_retraso"),
        F.when(
            F.col("p.fecha_entrega_cliente").isNotNull() & F.col("p.fecha_entrega_estimada").isNotNull(),
            F.when(F.to_date("p.fecha_entrega_cliente") <= F.to_date("p.fecha_entrega_estimada"), 1).otherwise(0)
        ).otherwise(F.lit(None).cast("int")).cast("int").alias("entrega_a_tiempo"),
        F.coalesce(F.col("a.cantidad_articulos"), F.lit(0).cast("bigint")).cast("bigint").alias("cantidad_articulos"),
        F.col("a.importe_articulos"),
        F.col("a.importe_flete"),
        (F.col("a.importe_articulos") + F.col("a.importe_flete")).cast("decimal(18,2)").alias("importe_total"),
        F.when(F.col("a.cantidad_articulos").isNotNull(), F.lit(1)).otherwise(F.lit(0)).cast("int").alias("tiene_detalle"),
        F.lit(1).cast("int").alias("indicador_pedido")
    )
)

guardar_hecho(fact_pedidos, "fact_pedidos", 99441)

In [0]:
fact_pagos = (
    pagos.alias("pg")
    .join(cabeceras.alias("c"), F.col("pg.id_pedido") == F.col("c.id_pedido"), "left")
    .select(
        F.col("pg.id_pedido"),
        F.col("pg.secuencia_pago"),
        fecha_key("c.fecha_compra").alias("id_fecha"),
        F.col("c.id_cliente_unico"),
        F.col("pg.metodo_pago"),
        F.col("c.estado_pedido"),
        F.col("pg.importe_pago"),
        F.col("pg.cuotas")
    )
)

guardar_hecho(fact_pagos, "fact_pagos", 103886)

In [0]:
pedidos_evaluados = spark.table(f"{catalogo}.golden.fact_pedidos").select(
    "id_pedido", "id_cliente_unico", "estado_pedido", "entrega_a_tiempo"
)

fact_resenas = (
    resenas.alias("r")
    .join(pedidos_evaluados.alias("p"), F.col("r.id_pedido") == F.col("p.id_pedido"), "left")
    .select(
        F.col("r.id_resena"),
        F.col("r.id_pedido"),
        fecha_key("r.fecha_creacion").alias("id_fecha"),
        F.col("p.id_cliente_unico"),
        F.col("p.estado_pedido"),
        F.col("r.calificacion"),
        F.col("r.fecha_creacion"),
        F.col("r.fecha_respuesta"),
        F.col("p.entrega_a_tiempo")
    )
)

guardar_hecho(fact_resenas, "fact_resenas", 99224)